In [7]:
!pip install -r requirements.txt

  Using cached torch-2.4.0-cp310-cp310-manylinux1_x86_64.whl.metadata (26 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64

In [39]:
import os, sys
import pathlib
import json

import torch
import pandas as pd
from transformers import AutoTokenizer
import torch.nn.functional as F

from data.dataset import build_id_to_text
from data.preprocessing import run_preprocessing
from model.train import (
    train_stage1, train_stage2,
    evaluate_epoch, DEFAULT_CFG, format_metrics,
    run_hpo, run_hpo_reranker, build_item_embedding_cache, tokenise_batch,
    build_cf_mappings,
)

from model.architecture import ContrastiveEncoder, TwoTowerModel
from model.reranker import CrossEncoderReranker, TwoTowerWithReranker, train_stage3
from model.metrics import evaluate_reranker, format_metrics

In [2]:
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
output_dir = pathlib.Path('checkpoints/')
output_dir.mkdir(exist_ok=True)

ENCODER = 'intfloat/multilingual-e5-large'
use_tower_hpo = False

# Data

In [4]:
os.makedirs('raw_data', exist_ok=True)

BASE_URL = 'https://huggingface.co/datasets/kdduha/shikimori-recsys/resolve/main'

for fname in ['anime.csv', 'genres.csv', 'users_rates.csv']:
    dest = f'raw_data/{fname}'
    if not os.path.exists(dest):
        !wget -q "{BASE_URL}/{fname}" -O "{dest}"
        print(f'Downloaded {fname}')
    else:
        print(f'Already have {fname}')

!ls -lh raw_data/

Already have anime.csv
Already have genres.csv
Already have users_rates.csv
total 15M
-rw-rw-r-- 1 s.senichev s.senichev 8.7M Mar  2 23:30 anime.csv
-rw-rw-r-- 1 s.senichev s.senichev 2.9K Mar  2 23:30 genres.csv
-rw-rw-r-- 1 s.senichev s.senichev 6.0M Mar  2 23:30 users_rates.csv


In [5]:
stats = run_preprocessing(
    data_dir='raw_data',
    output_dir='processed_data',
)
print(json.dumps(stats, indent=2))

14:29:29  INFO      Loading raw CSVs from raw_data
14:29:29  INFO      Raw sizes — anime: 9950  genres: 80  interactions: 67071
14:29:31  INFO      Anime processing done. Non-null text_input: 9950 / 9950
14:29:31  INFO      Saved anime_processed.parquet
14:29:31  INFO      Saved genre_vocab.json  (80 genres)
14:29:32  INFO      Dropped 0 rows with unparseable anime_id
14:29:32  INFO      Interactions — explicit (scored): 28129  implicit (watched, unscored): 38677  dropped (score=0, episodes=0): 265
14:29:32  INFO      Dropped 1515 interactions for unknown anime_id
14:29:32  INFO      Removed users with < 2 total interactions (explicit+implicit). Kept: 993 users  65143 rows
14:29:33  INFO      Final — 65143 rows (27390 explicit, 37753 implicit)  993 users  4182 items
14:29:33  INFO      Temporal split — train: 64151  val: 496  test: 496  (eligible users: 496 / 993)
14:29:33  WARNING   Temporal leakage: 137 val rows have created_at BEFORE user's latest train item (likely duplicate timest

{
  "n_anime_raw": 9950,
  "n_anime_processed": 9950,
  "n_genres": 80,
  "n_interactions_raw": 67071,
  "n_interactions_clean": 65143,
  "n_users": 993,
  "n_items": 4182,
  "train_size": 64151,
  "val_size": 496,
  "test_size": 496,
  "score_mean": 3.345,
  "score_std": 4.082,
  "density_pct": 1.5687
}


In [6]:
anime_df = pd.read_parquet('processed_data/anime_processed.parquet')
train_df = pd.read_parquet('processed_data/train_interactions.parquet')
val_df = pd.read_parquet('processed_data/val_interactions.parquet')
test_df = pd.read_parquet('processed_data/test_interactions.parquet')

print('Anime sample')
display(anime_df[['id','name','genre_names','rating_ordinal','score_global','text_input']].head(3))

print('\nInteraction sample')
display(train_df.head(3))

print(f'\nTrain: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')
# print(f'Score distribution:\n{train_df.score_raw.value_counts().sort_index()}')

Anime sample


,id,name,genre_names,rating_ordinal,score_global,text_input
0,52991,Sousou no Frieren,"[Adventure, Drama, Fantasy, Shounen]",2.0,0.921111,Sousou no Frieren (Провожающая в последний пут...
1,59978,Sousou no Frieren 2nd Season,"[Adventure, Drama, Fantasy, Shounen]",2.0,0.910000,Sousou no Frieren 2nd Season (Провожающая в по...
2,5114,Fullmetal Alchemist: Brotherhood,"[Action, Adventure, Drama, Fantasy, Shounen, M...",3.0,0.900000,Fullmetal Alchemist: Brotherhood (Стальной алх...



Interaction sample


,user_id,anime_id,score_raw,score_norm,rewatches,episodes,completion_rate,confidence,is_explicit,created_at
0,1721273,5114,9,0.888889,0,64,1.0,0.888889,True,2025-11-07 19:14:34+00:00
1,1721273,9253,10,1.000000,0,24,1.0,1.000000,True,2025-11-07 19:15:22+00:00
2,1721273,2001,10,1.000000,0,27,1.0,1.000000,True,2025-11-07 19:18:22+00:00



Train: 64,151  Val: 496  Test: 496


# Model

### Load checkpoint if it exists

In [7]:
# output_dir = pathlib.Path('checkpoints/final_model_v4_with_rerank_hpo')

# with open(output_dir / 'config.json') as f:
#     cfg = json.load(f)

# tokenizer = AutoTokenizer.from_pretrained(str(output_dir))

# # Rebuild CF mappings
# item_id_to_cf_idx, user_id_to_cf_idx, n_items_cf, n_users_cf = build_cf_mappings(anime_df, train_df)
# cf_dim = cfg.get('cf_dim', 0)
# if cf_dim == 0:
#     n_items_cf = 0
#     n_users_cf = 0

# model = TwoTowerModel(
#     encoder_name=ENCODER,
#     proj_dim=cfg['proj_dim'],
#     nhead=cfg['nhead'],
#     temperature=cfg['temperature'],
#     dropout=cfg['dropout'],
#     freeze_mode=cfg['freeze_mode'],
#     lora_rank=cfg['lora_rank'],
#     lora_alpha=cfg.get('lora_alpha', 16.0),
#     n_items=n_items_cf,
#     n_users=n_users_cf,
#     cf_dim=cf_dim,
#     user_tower_layers=cfg.get('user_tower_layers', 1),
# )

# state = torch.load(output_dir / 'model.pt', map_location=device)
# model.load_state_dict(state, strict=False)
# model.to(device)
# model.eval()
# print('Final model loaded')

## Train main model

### HPO

In [8]:
tokenizer = AutoTokenizer.from_pretrained(ENCODER)
hpo_path = output_dir / 'hpo_best_params.json'

if use_tower_hpo:
    print('Starting HPO')
    best_params = run_hpo(
        train_df=train_df,
        val_df=val_df,
        anime_df=anime_df,
        base_cfg=DEFAULT_CFG,
        output_dir=pathlib.Path('checkpoints'),
        device=device,
        n_trials=20,
        encoder_name=ENCODER,
    )
    
    with open(hpo_path) as f:
        hpo_data = json.load(f)
    cfg = {**DEFAULT_CFG, **hpo_data['best_params']}
    print(f'Using HPO params (NDCG@10={hpo_data["best_value"]:.4f})')

else:
    cfg = {**DEFAULT_CFG}
    print('Using default hyperparameters')

print(json.dumps(cfg, indent=2))

14:29:33  INFO      HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
14:29:33  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large/0dc5580a448e4284468b8909bae50fa925907bc5/config.json "HTTP/1.1 200 OK"
14:29:33  INFO      HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
14:29:33  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large/0dc5580a448e4284468b8909bae50fa925907bc5/tokenizer_config.json "HTTP/1.1 200 OK"
14:29:34  INFO      HTTP Request: GET https://huggingface.co/api/models/intfloat/multilingual-e5-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
14:29:34  INFO      HTTP Request: GET https://huggingface.co/api/models/intfloat/multilingual-e5-large/tr

Using default hyperparameters
{
  "s1_epochs": 3,
  "s1_batch_size": 32,
  "s1_grad_accum": 4,
  "s1_lr": 2e-05,
  "s1_warmup_steps": 100,
  "s1_max_length": 512,
  "s1_neg_per_user": 5,
  "s1_grad_clip": 1.0,
  "s2_epochs": 20,
  "s2_batch_size": 64,
  "s2_grad_accum": 4,
  "s2_lr": 0.0003,
  "s2_warmup_steps": 200,
  "s2_max_length": 512,
  "s2_max_history": 50,
  "s2_grad_clip": 1.0,
  "s2_encode_batch": 128,
  "proj_dim": 256,
  "nhead": 4,
  "temperature": 0.07,
  "dropout": 0.1,
  "weight_decay": 0.01,
  "lora_rank": 8,
  "lora_alpha": 16.0,
  "lora_dropout": 0.05,
  "freeze_mode": "lora",
  "pooling": "mean",
  "cf_dim": 128,
  "user_tower_layers": 2,
  "s2_hard_neg_k": 256,
  "eval_ks": [
    5,
    10,
    20
  ],
  "num_workers": 2,
  "s3_encoder": "distilbert-base-multilingual-cased",
  "s3_epochs": 1,
  "s3_batch_size": 16,
  "s3_grad_accum": 4,
  "s3_lr": 2e-05,
  "s3_warmup_steps": 50,
  "s3_max_length": 512,
  "s3_grad_clip": 1.0,
  "s3_neg_per_user": 3,
  "retrieval_k":

### Train Stage 1 (item tower)

In [9]:
s1_model = ContrastiveEncoder(
    encoder_name=ENCODER,
    proj_dim=cfg['proj_dim'],
    dropout=cfg['dropout'],
    freeze_mode=cfg['freeze_mode'],
    lora_rank=cfg['lora_rank'],
    lora_alpha=cfg.get('lora_alpha', 16.0),
)

s1_model = train_stage1(
    s1_model, tokenizer,
    train_df=train_df,
    val_df=val_df,
    anime_df=anime_df,
    cfg=cfg,
    output_dir=output_dir,
    device=device,
)

torch.save(s1_model.tower.state_dict(), output_dir / 'stage1_encoder.pt')
print('Stage 1 complete')

14:29:36  INFO      HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
14:29:36  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large/0dc5580a448e4284468b8909bae50fa925907bc5/config.json "HTTP/1.1 200 OK"
/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 792.33it/s, Materializing param=pooler.dense.weight]                               
XLMRobertaModel LOAD REPORT

[ItemTower] Applied LoRA to 48 linear layers
[ItemTower] Gradient checkpointing enabled


/home/s.senichev/anirec/model/train.py:115: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device.type == "cuda"))
/home/s.senichev/anirec/model/train.py:135: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == "cuda")):
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]
14:31:04  INFO      S1 E1/3  step 50/100  loss=0.1835  d_ap=0.2108  d_an=0.2232  lr=4.16e-06
14:32:23  INFO      S1 E1/3  step 100/100  loss=0.2132  d_

Stage 1 complete


### Train Stage 2 (user tower)

In [ ]:
# Build CF mappings
item_id_to_cf_idx, user_id_to_cf_idx, n_items_cf, n_users_cf = build_cf_mappings(anime_df, train_df)
cf_dim = cfg.get('cf_dim', 0)
if cf_dim == 0:
    n_items_cf = 0
    n_users_cf = 0

model = TwoTowerModel(
    encoder_name=ENCODER,
    proj_dim=cfg['proj_dim'],
    nhead=cfg['nhead'],
    temperature=cfg['temperature'],
    dropout=cfg['dropout'],
    freeze_mode=cfg['freeze_mode'],
    lora_rank=cfg['lora_rank'],
    lora_alpha=cfg.get('lora_alpha', 16.0),
    n_items=n_items_cf,
    n_users=n_users_cf,
    cf_dim=cf_dim,
    user_tower_layers=cfg.get('user_tower_layers', 1),
)

state = torch.load(output_dir / 'stage1_encoder.pt', map_location='cpu')
model.item_tower.load_state_dict(state, strict=False)
print('Stage 1 weights loaded into item tower')

model, best_val = train_stage2(
    model, tokenizer,
    train_df=train_df,
    val_df=val_df,
    anime_df=anime_df,
    cfg=cfg,
    output_dir=output_dir,
    device=device,
)

14:57:14  INFO      HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
14:57:14  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large/0dc5580a448e4284468b8909bae50fa925907bc5/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1134.95it/s, Materializing param=pooler.dense.weight]                               
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  war

[ItemTower] Applied LoRA to 48 linear layers
[ItemTower] Gradient checkpointing enabled


14:57:15  INFO      ============================================================
14:57:15  INFO      Stage 2: Two-tower training
14:57:15  INFO      ============================================================


Stage 1 weights loaded into item tower


/home/s.senichev/anirec/model/train.py:293: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device.type == "cuda"))
14:57:28  INFO        Building item embedding cache...
/home/s.senichev/anirec/model/train.py:203: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast("cuda"):
/home/s.senichev/anirec/model/train.py:344: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast("cuda"):
14:58:56  INFO      S2 E1/20  step 100/332  loss=4.1609  lr=6.38e-05


In [23]:
print('Best Val Metrics:', format_metrics(best_val))

Best Val Metrics: HR@10: 0.0282  HR@20: 0.0363  HR@5: 0.0181  MRR: 0.0150  NDCG@10: 0.0160  NDCG@20: 0.0181  NDCG@5: 0.0127


### Save model

In [45]:
final_dir = pathlib.Path('checkpoints/new_model_cf_v1_hpo')
final_dir.mkdir(exist_ok=True)

torch.save(model.state_dict(), final_dir / 'model.pt')
tokenizer.save_pretrained(str(final_dir))
with open(final_dir / 'config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'Model saved to {final_dir}')

Model saved to checkpoints/new_model_cf_v1_hpo


## Evaluate model

In [28]:
model.eval()

train_and_val = pd.concat([train_df, val_df], ignore_index=True)
test_metrics = evaluate_epoch(
    model, tokenizer, test_df, anime_df,
    cfg=cfg, device=device, ks=[10, 20, 50],
    train_df=train_and_val,
)

print('TEST RESULTS')
print(format_metrics(test_metrics))

/home/s.senichev/anirec/model/train.py:203: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == "cuda")):
/home/s.senichev/anirec/model/train.py:511: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == "cuda")):
18:17:25  INFO      Evaluation: 495 / 496 users have context embeddings


TEST RESULTS
HR@10: 0.0282  HR@20: 0.0323  HR@50: 0.0585  MRR: 0.0160  NDCG@10: 0.0169  NDCG@20: 0.0179  NDCG@50: 0.0230


## Reranker model

In [46]:
use_reranker_hpo = True

### Load checkpoint if it exists

In [47]:
# cfg = {**DEFAULT_CFG}

# s3_tokenizer = AutoTokenizer.from_pretrained(cfg["s3_encoder"])

# reranker_model = CrossEncoderReranker(encoder_name=cfg['s3_encoder'])
# reranker_model.load_state_dict(
#     torch.load('checkpoints/stage3_best.pt', map_location='cpu')
# )
# reranker_model.to(device).eval()
# print("Reranker loaded")

### Train reranker with HPO

In [ ]:
from model.train import run_hpo_reranker, DEFAULT_CFG

if use_reranker_hpo:
    best_s3_params = run_hpo_reranker(
        two_tower_model=model,
        train_df=train_df,
        val_df=val_df,
        anime_df=anime_df,
        base_cfg=cfg,
        output_dir=output_dir,
        device=device,
        n_trials=20,
    )
else:
    best_s3_params = {}

18:32:54  INFO      HTTP Request: HEAD https://huggingface.co/distilbert-base-multilingual-cased/resolve/main/config.json "HTTP/1.1 200 OK"
18:32:54  INFO      HTTP Request: HEAD https://huggingface.co/distilbert-base-multilingual-cased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
18:32:54  INFO      HTTP Request: GET https://huggingface.co/api/models/distilbert-base-multilingual-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
18:32:54  INFO      HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-multilingual-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
18:32:54  INFO      HTTP Request: GET https://huggingface.co/api/models/distilbert-base-multilingual-cased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
18:32:54  INFO      HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-multilingual-cased

In [ ]:
cfg_s3 = {**DEFAULT_CFG, **best_s3_params}

s3_tokenizer = AutoTokenizer.from_pretrained(cfg_s3["s3_encoder"])
reranker_model = train_stage3(
    CrossEncoderReranker(encoder_name=cfg['s3_encoder']),
    s3_tokenizer, train_df, val_df, anime_df,
    cfg=cfg_s3, output_dir=output_dir, device=device,
)

### Save reranker model

In [ ]:
torch.save(reranker_model.state_dict(), final_dir / 'reranker.pt')
s3_tokenizer.save_pretrained(str(final_dir / 'reranker_tokenizer'))

print(f'Reranker saved to {final_dir}')

In [55]:
recommender = TwoTowerWithReranker(
    two_tower=model,
    reranker=reranker_model,
    tokenizer=tokenizer,
    reranker_tokenizer=s3_tokenizer,
    anime_df=anime_df,
    device=device,
    item_id_to_cf_idx=item_id_to_cf_idx if model.has_cf else None,
    user_id_to_cf_idx=user_id_to_cf_idx if model.has_cf else None,
)

20:14:31  INFO      Building item embedding catalogue...
20:14:43  INFO      Catalogue ready: 9950 items × 256 dims


### Evaludate reranker

In [56]:
val_metrics = evaluate_reranker(
    recommender=recommender,
    holdout_df=val_df,
    train_df=train_df,
    ks=[10, 20, 50],
    retrieval_k=100,
)
print("RERANKER VAL METRICS")
print(format_metrics(val_metrics))

20:14:43  INFO      Evaluating reranker on 496 users (retrieval_k=100)
20:14:44  INFO        32 / 496 users evaluated...
20:14:48  INFO        352 / 496 users evaluated...
20:14:50  INFO      Retrieval recall: 37 / 496 users had target in candidates (7.46%)


RERANKER VAL METRICS
HR@10: 0.0202  HR@20: 0.0302  HR@50: 0.0585  MRR: 0.0960  NDCG@10: 0.0088  NDCG@20: 0.0112  NDCG@50: 0.0168  retrieval_recall: 0.0746


In [57]:
# Test set (context = train + val)
import pandas as pd
train_and_val = pd.concat([train_df, val_df], ignore_index=True)
test_metrics = evaluate_reranker(
    recommender=recommender,
    holdout_df=test_df,
    train_df=train_and_val,
    ks=[10, 20, 50],
    retrieval_k=100,
)
print("RERANKER TEST METRICS")
print(format_metrics(test_metrics))

20:14:50  INFO      Evaluating reranker on 496 users (retrieval_k=100)
20:14:50  INFO        32 / 496 users evaluated...
20:14:55  INFO        352 / 496 users evaluated...
20:14:57  INFO      Retrieval recall: 43 / 496 users had target in candidates (8.67%)


RERANKER TEST METRICS
HR@10: 0.0222  HR@20: 0.0383  HR@50: 0.0746  MRR: 0.1074  NDCG@10: 0.0106  NDCG@20: 0.0146  NDCG@50: 0.0216  retrieval_recall: 0.0867


# Load fully trained model

In [ ]:
final_dir = pathlib.Path('checkpoints/final_model_v4_with_rerank_hpo')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open(final_dir / 'config.json') as f:
    cfg = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(str(final_dir))

# Rebuild CF mappings for the loaded model
anime_df = pd.read_parquet('processed_data/anime_processed.parquet')
train_df = pd.read_parquet('processed_data/train_interactions.parquet')
item_id_to_cf_idx, user_id_to_cf_idx, n_items_cf, n_users_cf = build_cf_mappings(anime_df, train_df)
cf_dim = cfg.get('cf_dim', 0)
if cf_dim == 0:
    n_items_cf = 0
    n_users_cf = 0

model = TwoTowerModel(
    encoder_name='intfloat/multilingual-e5-base',
    proj_dim=cfg['proj_dim'], nhead=cfg['nhead'],
    temperature=cfg['temperature'], dropout=cfg['dropout'],
    freeze_mode=cfg['freeze_mode'], lora_rank=cfg['lora_rank'],
    lora_alpha=cfg.get('lora_alpha', 16.0),
    n_items=n_items_cf,
    n_users=n_users_cf,
    cf_dim=cf_dim,
    user_tower_layers=cfg.get('user_tower_layers', 1),
)
model.load_state_dict(torch.load(final_dir / 'model.pt', map_location='cpu'), strict=False)
model.to(device).eval()

s3_tokenizer = AutoTokenizer.from_pretrained(str(final_dir / 'reranker_tokenizer'))
reranker_model = CrossEncoderReranker(encoder_name=cfg['s3_encoder'])
reranker_model.load_state_dict(torch.load(final_dir / 'reranker.pt', map_location='cpu'))
reranker_model.to(device).eval()

print('Loaded')

# Inference 

In [58]:
# sample user

user_history = [
    (5114, 10),   # FMA Brotherhood
    (9253, 9),   # Steins;Gate
    (11061, 9),   # HxH 2011
    (37510, 8),   # Mob Psycho 100 II
    (31240, 3),   # Some disliked item (placeholder)
]
top_k = 20

## Test model without reranker

In [59]:
model.eval()

item_matrix, id_to_idx = build_item_embedding_cache(
    model, tokenizer, anime_df, cfg, device,
    item_id_to_cf_idx=item_id_to_cf_idx if model.has_cf else None,
)
idx_to_id = {v: k for k, v in id_to_idx.items()}

context_ids = [aid  for aid, _ in user_history if aid in id_to_idx]
context_scores = [s/10 for aid, s  in user_history if aid in id_to_idx]
watched_ids = set(context_ids)

ctx_idxs = torch.tensor([id_to_idx[a] for a in context_ids], dtype=torch.long)
ctx_embs = item_matrix[ctx_idxs].unsqueeze(0).to(device)
ctx_scores= torch.tensor([context_scores], dtype=torch.float).to(device)
ctx_mask  = torch.ones(1, len(context_ids), dtype=torch.bool).to(device)

with torch.no_grad():
    # No user_idx for anonymous/cold-start user — falls back to content-only
    user_vec = model.encode_user(ctx_embs, ctx_scores, ctx_mask)
    scores = (user_vec @ item_matrix.to(device).T).squeeze(0)

# Exclude watched items
watched_idxs = set(id_to_idx[a] for a in watched_ids if a in id_to_idx)
for idx in watched_idxs:
    scores[idx] = -1e9

top_idxs = scores.topk(top_k).indices.cpu().tolist()
top_ids = [idx_to_id[i] for i in top_idxs]

print(f'Top-{top_k} recommendations for this user:\n')
id_to_row = anime_df.set_index('id')
for rank, aid in enumerate(top_ids, 1):
    row = id_to_row.loc[aid]
    genres = ', '.join(row['genre_names'][:3])
    print(f'  {rank:2}. {row["name"]:<45} [{genres}]  score={row["score_global"]:.2f}')

/home/s.senichev/anirec/model/train.py:203: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == "cuda")):


Top-20 recommendations for this user:

   1. Kami no Tou                                   [Action, Adventure, Mystery]  score=0.73
   2. Nanbaka                                       [Action, Comedy, Drama]  score=0.70
   3. Ore Monogatari!!                              [Comedy, Romance, Shoujo]  score=0.77
   4. Ergo Proxy                                    [Mystery, Sci-Fi, Psychological]  score=0.77
   5. Devilman: Crybaby                             [Action, Avant Garde, Mythology]  score=0.75
   6. Mairimashita! Iruma-kun 3rd Season            [Comedy, Fantasy, School]  score=0.76
   7. Kobayashi-san Chi no Maid Dragon              [Slice of Life, Supernatural]  score=0.77
   8. Date A Live                                   [Action, Fantasy, Romance]  score=0.68
   9. Sakura-sou no Pet na Kanojo                   [Drama, Romance, School]  score=0.78
  10. Kemono Jihen                                  [Action, Mythology, Mystery]  score=0.71
  11. Nanatsu no Taizai: Imashime no Fu

## Test model with reranker

In [60]:
recs = recommender.recommend(user_history, top_k=top_k, retrieval_k=100)

print(f'Top- recommendations\n')
print(f'  {"#":<4} {"Title":<45} {"Reranker":>10} {"Retrieved":>10} {"Genres"}')
print('  ' + '-'*85)
for r in recs:
    genres = ', '.join(list(r['genres'])[:3]) if len(r['genres']) > 0 else '—'
    moved = r['two_tower_rank'] - r['rank']
    arrow = f'↑{moved}' if moved > 0 else (f'↓{-moved}' if moved < 0 else '=')
    print(f"  {r['rank']:<4} {r['name']:<45} {r['reranker_score']:>6.3f}  "
          f"#{r['two_tower_rank']:<3} {arrow:<5}  [{genres}]")

Top- recommendations

  #    Title                                           Reranker  Retrieved Genres
  -------------------------------------------------------------------------------------
  1    Overlord                                       0.822  #67  ↑66    [Action, Adventure, Fantasy]
  2    Nanatsu no Taizai: Imashime no Fukkatsu        0.815  #17  ↑15    [Action, Adventure, Fantasy]
  3    Fullmetal Alchemist: The Conqueror of Shamballa  0.810  #23  ↑20    [Action, Adventure, Drama]
  4    Juubee Ninpuuchou                              0.810  #97  ↑93    [Action, Adventure, Fantasy]
  5    Akira                                          0.808  #63  ↑58    [Action, Horror, Sci-Fi]
  6    Jin-Rou                                        0.808  #65  ↑59    [Action, Drama, Romance]
  7    Undead Unluck                                  0.797  #89  ↑82    [Action, Comedy, Shounen]
  8    Afro Samurai                                   0.790  #51  ↑43    [Action, Adventure, Samurai]
  9